In [1]:
version = "REPLACE_PACKAGE_VERSION"

# Assignment 2: Mining Itemsets (Part IV)

## Evaluating Frequent Itemsets

Even though we have found all the frequent itemsets, not all of them are interesting. In Part IV of this assignment, we will practice how to evaluate the frequent itemsets.

First, let's import the packages and dependencies that will be used in this part of assignment 2.

In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import MultiLabelBinarizer

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

from mads.lib.path import assets

**<span style="color:red">NOTE: These are all the imports we need to make for this assignment (Part IV). You should not make other imports in your submitted notebook. You will receive 0 points for the exercises if your solution includes additional imports.</span>**

People have developed various measurements of the interestingness of patterns. Most of them split the itemset into an antecedent item(set) and a consequent item(set), and then measure the correlation between the antecedent and the consequent. Let's try some of such measurements implemented by the `mlxtend.frequent_patterns.association_rules` API, which we have imported. For more information about the API, visit the [documentation](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/) of the `mlxtend` package.

Let's again use the shopping basket dataset as an example.

In [3]:
# Load data from files.
file_orders = assets.find("orders.csv.zip")
file_products = assets.find("products.csv.zip")
orders = pd.read_csv(file_orders, nrows=10000)
products = pd.read_csv(file_products)

# Group orders by order id and merge them into a list.
order_baskets = orders.groupby("order_id")["product_id"].apply(list)

# Convert the above pandas Series to a pandas DataFrame.
order_baskets = order_baskets.to_frame(name="products_id")

# Create the name map for later reference.
product_name_map = dict(zip(products["product_id"], products["product_name"]))

# Create the matrix like Part 1.
mlb = MultiLabelBinarizer()
data = mlb.fit_transform(order_baskets["products_id"]).astype(bool)
prod_matrix = pd.DataFrame(
    data=data, index=order_baskets.index, columns=mlb.classes_
)
prod_popularity = prod_matrix.sum(axis=0)
prod_matrix = prod_matrix.astype(bool)
top_prods = prod_popularity.sort_values(ascending=False).head(100).index
prod_matrix = prod_matrix[top_prods]
prod_matrix.columns = [product_name_map[x] for x in prod_matrix.columns]
prod_matrix.head()

,Banana,Bag of Organic Bananas,Organic Strawberries,Organic Baby Spinach,Organic Hass Avocado,Organic Avocado,Strawberries,Large Lemon,Organic Yellow Onion,Organic Raspberries,...,Organic Lacinato (Dinosaur) Kale,Sharp Cheddar Cheese,Organic Romaine Lettuce,Green Onions,Small Hass Avocado,Marinara Sauce,Jalapeno Peppers,Organic Red Delicious Apple,Sparkling Lemon Water,Sparkling Natural Mineral Water
order_id,,,,,,,,,,,,,,,,,,,,,
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,False,True,False,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [4]:
prod_frequent_itemsets = apriori(prod_matrix, min_support=0.005, use_colnames=True)

In [5]:
interestingness_measurements = association_rules(prod_frequent_itemsets, metric="lift", min_threshold=0)
interestingness_measurements.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Banana}),frozenset({Organic Strawberries}),0.159672,0.074719,0.012282,0.076923,1.029505,1.0,0.000352,1.002388,0.034105,0.055300,0.002383,0.120653
1,frozenset({Organic Strawberries}),frozenset({Banana}),0.074719,0.159672,0.012282,0.164384,1.029505,1.0,0.000352,1.005638,0.030973,0.055300,0.005606,0.120653
2,frozenset({Banana}),frozenset({Organic Baby Spinach}),0.159672,0.071648,0.024565,0.153846,2.147253,1.0,0.013125,1.097143,0.635810,0.118812,0.088542,0.248352
3,frozenset({Organic Baby Spinach}),frozenset({Banana}),0.071648,0.159672,0.024565,0.342857,2.147253,1.0,0.013125,1.278759,0.575524,0.118812,0.217992,0.248352
4,frozenset({Banana}),frozenset({Organic Hass Avocado}),0.159672,0.069601,0.008188,0.051282,0.736802,1.0,-0.002925,0.980691,-0.298291,0.037037,-0.019689,0.084465


In the returned data frame, each row examines one (antecedent -> consequent) pair. *Antecedent support* and *consequent support* measure *P*(antecedent) and *P*(consequent), while *support* measures *P*(antecedent, consequent). In fact, these three values help us characterize the $2\times2$ contingency table, as illustrated in the following table:
 
 |           |              |       |              | 
 -----------:|:------------:|-------|---------------
 |           |    X = 1     | X = 0 |   sum(row)   |
 |     Y = 1 |   `support`    |       | `cons_support` |
 |     Y = 0 |              |       |              |
 | sum(col.) | `ante_support` |       |              |

Most interestingness measurements, including the four shown in the data frame (*confidence*, *lift*, *leverage*, and *conviction*), can be derived from the three support values. For example, $$\text{confidence}=\frac{\text{support}}{\text{antecedent\_support}},$$ and $$\text{lift} =\frac{\text{confidence}}{\text{consequent\_support}}=\frac{\text{support}}{\text{antecedent\_support} * \text{consequent\_support}}$$

### Exercise 5. (15 pts)
In this exercise, we are going to implement another interestingness
measurement, the (full) mutual information, and add a 'mutual information'
column to the data frame. The measurement is defined as

$$I(X;Y)=\sum_{x\in\mathcal{X}}\sum_{y\in\mathcal{Y}} P(X=x,
Y=y)\log_2\frac{P(X=x,Y=y)}{P(X=x)P(Y=y)}.$$

(*The [mutual information](https://en.wikipedia.org/wiki/Mutual_information)
quantifies the "amount of information obtained from one random variable by
observing the other random variable.*)

Note that the logorithm requirest that the joint probability $P(X=x, Y=y) > 0$,
which does not hold for some $(x, y)$. However, since we know that when $P(X=x,
Y=y) = 0$, it would not contribute to the sum, you may assume $P(X=x,
Y=y)\log_2\frac{P(X=x,Y=y)}{P(X=x)P(P=y)} = 0$ in that case. 

$x$, $y$ are possible values of $X$ and $Y$; in the case of appearance or
absence of an item, 1 or 0. Therefore, we need to consider all possible
combinations of $x$ and $y$, that is, $(X=1, Y=1)$, $(X=1, Y=0)$, $(X=0, Y=1)$,
$(X=0, Y=0)$.

Please complete the following function that uses the three support values to
compute the mutual information. All the three parameters are in [0, 1], and you
can assume the validity of the input. **Use 2 as the log base.** We have
created some auxilary variables for you, each represent a joint or marginal
(let $X$ denote the antecedent item and $Y$ denote the consequent item)
probability.

In [6]:
def mi(antecedent_support, consequent_support, support):

    # get marginal probabilities for X and Y
    # X=1 means antecedent occurs, X=0 means it does not
    # Y=1 means consequent occurs, Y=0 means it does not
    px1 = antecedent_support
    px0 = 1 - antecedent_support
    py1 = consequent_support
    py0 = 1 - consequent_support
    
    # compute all four joint probabilities
    px1y1 = support
    px1y0 = px1 - px1y1
    px0y1 = py1 - px1y1
    px0y0 = 1 - px1 - py1 + px1y1

    # start MI at 0 and add each valid term
    mutual_information = 0

    # add the (X=1, Y=1) term if its probability is positive
    if px1y1 > 0:
        mutual_information += px1y1 * np.log2(px1y1 / (px1 * py1))
    # add the (X=1, Y=0) term if its probability is positive
    if px1y0 > 0:
        mutual_information += px1y0 * np.log2(px1y0 / (px1 * py0))
    # add the (X=0, Y=1) term if its probability is positive
    if px0y1 > 0:
        mutual_information += px0y1 * np.log2(px0y1 / (px0 * py1))
    # add the (X=0, Y=0) term if its probability is positive
    if px0y0 > 0:
        mutual_information += px0y0 * np.log2(px0y0 / (px0 * py0))

    # return the final MI value
    return mutual_information

In [7]:
# This code block tests whether the `mi` function work as expected.
# We hide some tests, so passing all the displayed assertions does not guarantee the bonus points.

assert np.abs(mi(0.6, 0.75, 0.4) - 0.04287484674660057) < 1e-8
assert np.abs(mi(0.5, 0.5, 0.25) - 0) < 1e-8

# If you fail the following assertion, double check if your function 
# handles the scenarios in which a joint probability is zero.
assert np.abs(mi(0.5, 0.5, 0.5) - 1) < 1e-8



*The for loop of Python is very slow. In real-world scenarios, we will use
vectorization to speed up the computation. For the `mi()` function, we can also
vectorize it to be `mi_vec()`. Now, suppose the inputs are all arrays in shape
of `(n,)`, how can you implement this function?*

Hint:
- The returned `mutual_information` should also be in shape of `(n,)`.
- Use `np.where()` function to replace `if`-condition. See
  [`numpy.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html)
  document.

In [8]:
def mi_vec(antecedent_support: np.ndarray, consequent_support: np.ndarray,
           support: np.ndarray):

    # get marginal probabilities for all rules at once
    px1 = antecedent_support
    px0 = 1 - antecedent_support
    py1 = consequent_support
    py0 = 1 - consequent_support

    # compute the four joint probabilities for all rules
    px1y1 = support
    px1y0 = px1 - px1y1
    px0y1 = py1 - px1y1
    px0y0 = 1 - px1 - py1 + px1y1

    # start an array of zeros to store MI values
    mutual_information = np.zeros_like(support)

    # add the (X=1, Y=1) contribution where probability is positive
    mask = px1y1 > 0
    mutual_information[mask] += px1y1[mask] * np.log2(px1y1[mask] / (px1[mask] * py1[mask]))

    # dd the (X=1, Y=0) contribution where probability is positive
    mask = px1y0 > 0
    mutual_information[mask] += px1y0[mask] * np.log2(px1y0[mask] / (px1[mask] * py0[mask]))

    # add the (X=0, Y=1) contribution where probability is positive
    mask = px0y1 > 0
    mutual_information[mask] += px0y1[mask] * np.log2(px0y1[mask] / (px0[mask] * py1[mask]))

    # # add the (X=0, Y=0) contribution where probability is positive
    mask = px0y0 > 0
    mutual_information[mask] += px0y0[mask] * np.log2(px0y0[mask] / (px0[mask] * py0[mask]))

    # return MI values for all rules
    return mutual_information

Mutual information is a classical measure of interestingness. We encourage you to think about the following questions (not graded):
1. What is the maximum of mutual information in this setting? How to reach it?
2. What is the minimum of mutual information in this setting? How to reach it?

What does the max/min value imply?

- **Maximum mutual information:** The maximum is 1 bit in this binary setting. It is reached when the antecedent and consequent are **perfectly dependent**, so knowing one completely determines the other, such as $Y = X$ or $Y = 1 - X$, with balanced probabilities like $P(X = 1) = P(Y = 1) = 0.5$.

- **Minimum mutual information:** The minimum is 0. It is reached when the antecedent and consequent are **independent**, meaning $P(X,Y) = P(X)P(Y)$, or equivalently the joint support equals the product of the individual supports.

- **What the max/min imply:** A **maximum** value means the rule is maximally informative: knowing the antecedent removes all uncertainty about the consequent. A **minimum** value of 0 means the rule carries no information: knowing the antecedent tells you nothing extra about the consequent.

With the `mi` function, we can now compute the mutual information for each (antecedent -> consequent) pair and attach it to the data frame. Does the result make sense?

In [9]:
interestingness_measurements['mi'] = \
    interestingness_measurements.apply(lambda pair: mi(pair['antecedent support'], 
                                              pair['consequent support'], 
                                              pair['support']),
                                       axis=1)
interestingness_measurements.sort_values('mi', ascending=False).head(n=5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,mi
260,frozenset({Strawberries}),frozenset({Raspberries}),0.053224,0.026612,0.010235,0.192308,7.226331,1.0,0.008819,1.205147,0.910054,0.147059,0.170226,0.288462,0.020237
261,frozenset({Raspberries}),frozenset({Strawberries}),0.026612,0.053224,0.010235,0.384615,7.226331,1.0,0.008819,1.538511,0.885174,0.147059,0.350021,0.288462,0.020237
92,frozenset({Bag of Organic Bananas}),frozenset({Organic Hass Avocado}),0.121801,0.069601,0.023541,0.193277,2.776940,1.0,0.015064,1.153307,0.728641,0.140244,0.132928,0.265756,0.017598
93,frozenset({Organic Hass Avocado}),frozenset({Bag of Organic Bananas}),0.069601,0.121801,0.023541,0.338235,2.776940,1.0,0.015064,1.327056,0.687760,0.140244,0.246452,0.265756,0.017598
282,frozenset({Organic Zucchini}),frozenset({Organic Garlic}),0.029683,0.035824,0.008188,0.275862,7.700493,1.0,0.007125,1.331481,0.896756,0.142857,0.248957,0.252217,0.016409


Let's benchmark the speed of two different implementations. We scale the data by 1000 times. Can you find a significant acceleration in the vectorized version of `mi_vec()`?

In [10]:
benchmark_data = pd.concat([interestingness_measurements] * 1000,
                           ignore_index=True)
print(f"benchmark_data contains {len(benchmark_data)} entries.")

start = time.time()
result1 = benchmark_data.apply(lambda pair: mi(
    pair['antecedent support'], pair['consequent support'], pair['support']),
                                             axis=1)
end = time.time()
print(f"mi() takes {end-start:.4f} seconds.")

start = time.time()
result2 = mi_vec(benchmark_data['antecedent support'],
                 benchmark_data['consequent support'],
                 benchmark_data['support'])
end = time.time()
print(f"mi_vec() takes {end-start:.4f} seconds.")

assert abs(result1 - result2).max() < 1e-8, "mi_vec() and mi() produce inconsistent results."

benchmark_data contains 332000 entries.
mi() takes 3.7906 seconds.
mi_vec() takes 0.0205 seconds.
